# GPT + MolE: Table 1 over 5 full-pipeline trials (GPU)

Each trial *k* repeats the whole protocol for all 9 Table 1 rows: hyperparameter search seeded with *k* (5 configs x 3 fits, the same configs for every row so rows stay paired), selection on mean validation PR-AUC, then one test run with training seed *k*. 45 runs in total.

**Setup**
1. Upload `kaggle/gpt_mole_bundle.zip` as a new Kaggle dataset (Kaggle unpacks it).
2. In this notebook: *Add Input* -> that dataset.
3. *Settings*: Accelerator **GPU** (T4 or P100), Internet **on** (only needed if `rdkit` is missing).
4. Run cells 1-4 interactively to confirm the GPU works and see the timing of one run, then use **Save Version -> Save & Run All** so the full run continues with the browser closed (12 h session limit).

If a session ends before all trials finish: attach this notebook's previous version output as an input; cell 3 copies its finished runs back and `--resume` skips them. Every row in one table must come from the same device, so do not mix these GPU runs with CPU runs.

In [ ]:
# 1. Confirm XGBoost really trains on the GPU. XGBoost silently falls back to CPU
#    when no GPU is visible, so check for that warning rather than trusting the setting.
import subprocess, sys, warnings
import numpy as np
import xgboost as xgb

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| xgboost", xgb.__version__)
assert int(xgb.__version__.split(".")[0]) >= 2, "device='cuda' needs xgboost >= 2"

X = np.random.rand(2000, 20)
y = (X[:, 0] > 0.5).astype(int)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    xgb.train({"tree_method": "hist", "device": "cuda", "objective": "binary:logistic"},
              xgb.DMatrix(X, label=y), 10)
fallback = [str(w.message) for w in caught if "GPU" in str(w.message)]
assert not fallback, f"XGBoost fell back to CPU - turn on a GPU accelerator.\n{fallback}"
print("GPU training OK")

In [ ]:
# 2. Dependencies. build_train.py imports rdkit (unused by these runs) and reads the
#    SF3 workbook with openpyxl; install either only if the image lacks it.
import importlib.util

for module, package in [("rdkit", "rdkit"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import pandas, sklearn
print("pandas", pandas.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
# 3. Copy the bundle to a writable folder (inputs are read-only) and work from there.
#    Also restores finished runs from a previous version's output, if one is attached.
import glob, os, shutil

hits = glob.glob("/kaggle/input/**/notebooks/build_train.py", recursive=True)
assert hits, "Bundle not found - attach the dataset made from gpt_mole_bundle.zip."
BUNDLE = os.path.dirname(os.path.dirname(hits[0]))
WORK = "/kaggle/working/gpt_mole"
if not os.path.exists(WORK):
    shutil.copytree(BUNDLE, WORK)
os.chdir(WORK)
print("bundle:", BUNDLE, "-> working in", os.getcwd())
if os.path.exists("MANIFEST.txt"):
    print(open("MANIFEST.txt").read())

# Kaggle decompresses .gz files inside an uploaded dataset, so x.tsv.gz arrives as
# x.tsv. Re-compress them to the names the scripts expect.
import gzip
sys.path.insert(0, "scripts")
import run_table1
needed = [run_table1.SF3, run_table1.SPLIT, run_table1.MOLE, run_table1.PRESTWICK,
          run_table1.CHEMDESC, *run_table1.FREEFORM.values()]
for path in needed:
    if path.endswith(".gz") and not os.path.exists(path) and os.path.exists(path[:-3]):
        with open(path[:-3], "rb") as src, gzip.open(path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        print("re-compressed", path)
missing = [p for p in needed if not os.path.exists(p)]
assert not missing, f"Missing inputs: {missing}"
print(f"all {len(needed)} inputs present")

OUT_ROOT = "results/table1_trials"
previous = [d for d in glob.glob("/kaggle/input/**/table1_trials", recursive=True)
            if not d.startswith(BUNDLE)]
for d in previous:
    shutil.copytree(d, OUT_ROOT, dirs_exist_ok=True)
    print("restored finished runs from", d)

In [ ]:
# 4. One run first (trial 0, MolE only) to check the pipeline and time a run.
#    The full run below skips it via --resume.
!python -u scripts/run_table1.py --trials 0 --only mole_only --out-root {OUT_ROOT} --device cuda --resume
assert _exit_code == 0, "run failed - see the output above and results/table1_trials/trial_0/mole_only/run.log"

In [ ]:
# 5. Full run: 5 trials x 9 rows. Set TRIALS to a subset (e.g. "0,1,2") to split
#    the work across sessions.
TRIALS = "0,1,2,3,4"
!python -u scripts/run_table1.py --trials {TRIALS} --out-root {OUT_ROOT} --device cuda --resume
assert _exit_code == 0, "some runs failed - see the list above and each run.log"

In [ ]:
# 6. Every run must have trained on the GPU; a CPU fallback would mix devices.
logs = glob.glob(f"{OUT_ROOT}/trial_*/*/run.log")
cpu_runs = [p for p in logs if "No visible GPU" in open(p, encoding="utf-8", errors="ignore").read()]
print(f"{len(logs)} run logs checked")
assert logs, "No run logs found - the runs above did not start."
assert not cpu_runs, f"These runs fell back to CPU:\n" + "\n".join(cpu_runs)
print("all runs on GPU")

In [ ]:
# 7. Aggregate: mean +/- sd over trials per row -> table1.csv and RESULTS_MANIFEST.csv.
!python scripts/aggregate_table1.py --root {OUT_ROOT}
assert os.path.exists(f"{OUT_ROOT}/table1.csv"), "aggregation produced no table1.csv"

In [ ]:
# 8. Paper-ready view: mean +/- sd, validation next to test.
import pandas as pd

t = pd.read_csv(f"{OUT_ROOT}/table1.csv")
fmt = lambda m, s: f"{m:.4f} +/- {s:.4f}" if pd.notna(s) and s != "" else f"{m:.4f}"
view = pd.DataFrame({
    "Representation": t["row"],
    "trials": t["n_runs"],
    "Val PR-AUC (per-strain)": [fmt(m, s) for m, s in zip(t.val_pr_auc_mean, t.val_pr_auc_sd)],
    "Test PR-AUC (per-strain)": [fmt(m, s) for m, s in zip(t.test_pr_auc_mean, t.test_pr_auc_sd)],
    "Test PR-AUC (pooled)": [fmt(m, s) for m, s in zip(t.test_pooled_pr_auc_mean, t.test_pooled_pr_auc_sd)],
})
view

In [ ]:
# 9. Package everything for download (Output tab -> table1_trials_results.zip).
#    The raw folder also stays in /kaggle/working/gpt_mole/results/table1_trials,
#    which is what cell 3 restores from in a later session.
archive = shutil.make_archive("/kaggle/working/table1_trials_results", "zip", OUT_ROOT)
print(archive, f"{os.path.getsize(archive) / 1024 ** 2:.1f} MB")